In [8]:
import pandas as pd

# Read the CSV file
file_path = 'Data/data_preprocessed/predicition_data/boiler_data_for_prediction.csv'  # Replace with your actual file path
df = pd.read_csv(file_path)
# Filter rows where 'value' column > 100
filtered_df = df[df['value'] >50]

# Display the result
print(filtered_df)


Empty DataFrame
Columns: [measurement, appliance, value, timestamp]
Index: []


In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# --- Step 1: Load your appliance power data ---
file_path = "Data\data_preprocessed\predicition_data\washing_machine_data_for_prediction.csv"  # Replace with your actual file path
df = pd.read_csv(file_path)  # should contain 'timestamp' and 'power'
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Optional: smooth short spikes or noise
df['power_smooth'] = df['value'].rolling(window=3, center=True).mean().fillna(method='bfill').fillna(method='ffill')

# --- Step 2: Reshape power data for clustering ---
X = df['power_smooth'].values.reshape(-1, 1)

# --- Step 3: Apply KMeans with 2 clusters (OFF/ON) ---
kmeans = KMeans(n_clusters=2, random_state=0)
df['cluster'] = kmeans.fit_predict(X)

# --- Step 4: Map cluster labels to states ---
# Assume lower mean = OFF, higher mean = ON
cluster_means = df.groupby('cluster')['power_smooth'].mean()
on_cluster = cluster_means.idxmax()
df['state'] = df['cluster'].apply(lambda c: 'ON' if c == on_cluster else 'OFF')

# Optional: binary column
df['state_binary'] = df['state'].map({'OFF': 0, 'ON': 1})

# --- Step 5: Save results ---
df.to_csv("appliance_states_kmeans.csv", index=False)

# --- Step 6: Plot to visualize (optional) ---
plt.figure(figsize=(12, 4))
plt.plot(df['timestamp'], df['power_smooth'], label='Power')
plt.fill_between(df['timestamp'], 0, df['power_smooth'], where=df['state_binary']==1, color='green', alpha=0.3, label='ON')
plt.legend()
plt.title("Appliance Power and Detected States (KMeans)")
plt.xlabel("Time")
plt.ylabel("Power (W)")
plt.tight_layout()
plt.show()


,measurement,appliance,value,timestamp
0,Electricity,washing_machine,0.000,1661990400
1,Electricity,washing_machine,0.000,1661990430
2,Electricity,washing_machine,0.000,1661990460
3,Electricity,washing_machine,0.000,1661990490
4,Electricity,washing_machine,0.000,1661990520
...,...,...,...,...
1041875,Electricity,washing_machine,0.696,1693526250
1041876,Electricity,washing_machine,0.988,1693526280
1041877,Electricity,washing_machine,0.923,1693526310
1041878,Electricity,washing_machine,0.800,1693526340
